<a href="https://colab.research.google.com/github/01Americonetto/delivery-intent-classifier/blob/main/delivery_intent_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install -q transformers torch
print("Dependências instaladas com sucesso.")


Dependências instaladas com sucesso.


In [21]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device_map="auto"
)

candidate_labels = [
    "atraso na entrega",
    "item faltando",
    "problema de pagamento",
    "cancelamento",
    "duvida geral"
]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [22]:
def route_support_intent(text: str) -> dict:
    """
    Classifica a intencao operacional de uma mensagem de suporte.
    """
    output = classifier(
        text,
        candidate_labels=candidate_labels,
        hypothesis_template="O problema relatado neste atendimento e sobre {}."
    )

    return {
        "mensagem": text,
        "intencao_predita": output["labels"][0],
        "score_confianca": float(output["scores"][0])
    }

In [23]:
chamados_teste = [
    "O entregador ja passou do prazo limite e o pacote nao chega de jeito nenhum.",
    "Faltou o refrigerante que estava incluido no combo do restaurante.",
    "Meu pagamento foi cobrado duas vezes no extrato do cartao.",
    "Gostaria de cancelar o pedido porque o prazo estourou.",
    "Qual e o horario de atendimento do suporte?"
]

print("--- RELATORIO DE TRIAGEM DE INTENCOES ---")
for chamado in chamados_teste:
    resultado = route_support_intent(chamado)
    print(f"Mensagem: '{resultado['mensagem']}'")
    print(f"-> Intencao: {resultado['intencao_predita']} (Confianca: {resultado['score_confianca']:.4f})\n")

--- RELATORIO DE TRIAGEM DE INTENCOES ---
Mensagem: 'O entregador ja passou do prazo limite e o pacote nao chega de jeito nenhum.'
-> Intencao: atraso na entrega (Confianca: 0.7341)

Mensagem: 'Faltou o refrigerante que estava incluido no combo do restaurante.'
-> Intencao: item faltando (Confianca: 0.9447)

Mensagem: 'Meu pagamento foi cobrado duas vezes no extrato do cartao.'
-> Intencao: problema de pagamento (Confianca: 0.9963)

Mensagem: 'Gostaria de cancelar o pedido porque o prazo estourou.'
-> Intencao: cancelamento (Confianca: 0.6158)

Mensagem: 'Qual e o horario de atendimento do suporte?'
-> Intencao: item faltando (Confianca: 0.3965)

